# Forecast benchmarks

Chronological train/validation/test split and naive price-forecast benchmarks.


## Why chronology matters

Time-series data must be split in time order. Randomly mixing earlier and later rows would let a model learn from information that would not yet have existed at the time of a historical decision.

The split uses:

- Training: 2015–2022
- Validation: 2023, for selecting an approach
- Final test: 2024, reserved for the final comparison
- Holdout: 2025, excluded here because this download changes to 15-minute resolution

All timestamps are handled in UTC. That gives a continuous hourly index and avoids daylight-saving-time ambiguity while developing the first forecast benchmarks.

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DATA_PATH = Path("data/belgium_day_ahead_2015_2025.csv")
NORMALISED_DATA_PATH = Path("data/belgium_day_ahead_hourly_2015_2024.csv")

raw_prices = pd.read_csv(RAW_DATA_PATH)
raw_prices.head()


,ts_utc,currency,eur_per_mwh
0,2015-01-04 23:00:00+00:00,NaN,36.56
1,2015-01-05 00:00:00+00:00,NaN,34.94
2,2015-01-05 01:00:00+00:00,NaN,32.19
3,2015-01-05 02:00:00+00:00,NaN,28.05
4,2015-01-05 03:00:00+00:00,NaN,28.04


## Standardise and select the hourly history


In [10]:
prices = raw_prices.rename(
    columns={"ts_utc": "timestamp", "eur_per_mwh": "price_eur_per_mwh"}
).copy()

prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True, errors="raise")
prices["price_eur_per_mwh"] = pd.to_numeric(
    prices["price_eur_per_mwh"], errors="raise"
)

# 2025 changes to quarter-hourly resolution, so keep the uniform hourly history.
hourly_prices = prices.loc[
    prices["timestamp"].dt.year <= 2024,
    ["timestamp", "price_eur_per_mwh"],
].sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

hourly_prices.head()


,timestamp,price_eur_per_mwh
0,2015-01-04 23:00:00+00:00,36.56
1,2015-01-05 00:00:00+00:00,34.94
2,2015-01-05 01:00:00+00:00,32.19
3,2015-01-05 02:00:00+00:00,28.05
4,2015-01-05 03:00:00+00:00,28.04


## Data-quality checks


In [11]:
assert hourly_prices["timestamp"].is_monotonic_increasing
assert not hourly_prices["timestamp"].duplicated().any()
assert hourly_prices["price_eur_per_mwh"].notna().all()

expected_timestamps = pd.date_range(
    hourly_prices["timestamp"].min(),
    hourly_prices["timestamp"].max(),
    freq="h",
    tz="UTC",
)
missing_timestamps = expected_timestamps.difference(hourly_prices["timestamp"])
assert len(missing_timestamps) == 0

print(f"Rows: {len(hourly_prices):,}")
print(f"Coverage: {hourly_prices['timestamp'].min()} to {hourly_prices['timestamp'].max()}")
print(f"Missing hourly timestamps: {len(missing_timestamps)}")
print(
    "Price range: "
    f"{hourly_prices['price_eur_per_mwh'].min():.2f} to "
    f"{hourly_prices['price_eur_per_mwh'].max():.2f} EUR/MWh"
)


Rows: 87,577
Coverage: 2015-01-04 23:00:00+00:00 to 2024-12-31 23:00:00+00:00
Missing hourly timestamps: 0
Price range: -500.00 to 871.00 EUR/MWh


## Preserve the normalised dataset


In [12]:
hourly_prices.to_csv(NORMALISED_DATA_PATH, index=False)
print(f"Saved {len(hourly_prices):,} hourly rows to {NORMALISED_DATA_PATH}")


Saved 87,577 hourly rows to data/belgium_day_ahead_hourly_2015_2024.csv


## Create chronological splits


In [13]:
TRAIN_END = pd.Timestamp("2023-01-01", tz="UTC")
VALIDATION_END = pd.Timestamp("2024-01-01", tz="UTC")
TEST_END = pd.Timestamp("2025-01-01", tz="UTC")

train = hourly_prices.loc[hourly_prices["timestamp"] < TRAIN_END].copy()
validation = hourly_prices.loc[
    (hourly_prices["timestamp"] >= TRAIN_END)
    & (hourly_prices["timestamp"] < VALIDATION_END)
].copy()
test = hourly_prices.loc[
    (hourly_prices["timestamp"] >= VALIDATION_END)
    & (hourly_prices["timestamp"] < TEST_END)
].copy()

assert train["timestamp"].max() < validation["timestamp"].min()
assert validation["timestamp"].max() < test["timestamp"].min()
assert len(validation) == 365 * 24
assert len(test) == 366 * 24  # 2024 is a leap year.

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "start": [train["timestamp"].min(), validation["timestamp"].min(), test["timestamp"].min()],
        "end": [train["timestamp"].max(), validation["timestamp"].max(), test["timestamp"].max()],
    }
)
split_summary


,split,rows,start,end
0,train,70033,2015-01-04 23:00:00+00:00,2022-12-31 23:00:00+00:00
1,validation,8760,2023-01-01 00:00:00+00:00,2023-12-31 23:00:00+00:00
2,test,8784,2024-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00


## Forecast benchmarks

Two transparent benchmarks are used:

- **Yesterday:** predict each price using the price 24 hours earlier.
- **Last week:** predict each price using the price 168 hours earlier.

At prediction time, both values are historical observations, so neither benchmark leaks future target prices. These are intentionally simple; a later model has to beat them on the validation period before it earns a place in the final comparison.

In [14]:
forecast_frame = hourly_prices.copy()
forecast_frame["forecast_yesterday_eur_per_mwh"] = forecast_frame[
    "price_eur_per_mwh"
].shift(24)
forecast_frame["forecast_last_week_eur_per_mwh"] = forecast_frame[
    "price_eur_per_mwh"
].shift(7 * 24)

forecast_frame.head(170).tail(5)


,timestamp,price_eur_per_mwh,forecast_yesterday_eur_per_mwh,forecast_last_week_eur_per_mwh
165,2015-01-11 20:00:00+00:00,34.06,35.00,NaN
166,2015-01-11 21:00:00+00:00,33.00,34.96,NaN
167,2015-01-11 22:00:00+00:00,32.19,32.91,NaN
168,2015-01-11 23:00:00+00:00,27.59,26.95,36.56
169,2015-01-12 00:00:00+00:00,26.81,24.22,34.94


## Evaluate error


In [15]:
def forecast_metrics(frame: pd.DataFrame, forecast_column: str) -> dict:
    """Calculate point-forecast error after excluding unavailable initial lags."""
    scored = frame.dropna(subset=[forecast_column]).copy()
    error = scored["price_eur_per_mwh"] - scored[forecast_column]
    return {
        "observations": len(scored),
        "mae_eur_per_mwh": error.abs().mean(),
        "rmse_eur_per_mwh": np.sqrt((error ** 2).mean()),
    }

validation_frame = forecast_frame.loc[
    (forecast_frame["timestamp"] >= TRAIN_END)
    & (forecast_frame["timestamp"] < VALIDATION_END)
]
test_frame = forecast_frame.loc[
    (forecast_frame["timestamp"] >= VALIDATION_END)
    & (forecast_frame["timestamp"] < TEST_END)
]

benchmark_columns = {
    "Same hour yesterday": "forecast_yesterday_eur_per_mwh",
    "Same hour last week": "forecast_last_week_eur_per_mwh",
}

rows = []
for split_name, frame in {"validation_2023": validation_frame, "test_2024": test_frame}.items():
    for model_name, forecast_column in benchmark_columns.items():
        rows.append(
            {
                "split": split_name,
                "benchmark": model_name,
                **forecast_metrics(frame, forecast_column),
            }
        )

benchmark_results = pd.DataFrame(rows)
benchmark_results[["observations", "mae_eur_per_mwh", "rmse_eur_per_mwh"]] = (
    benchmark_results[["observations", "mae_eur_per_mwh", "rmse_eur_per_mwh"]].round(2)
)
benchmark_results


,split,benchmark,observations,mae_eur_per_mwh,rmse_eur_per_mwh
0,validation_2023,Same hour yesterday,8760,23.76,34.56
1,validation_2023,Same hour last week,8760,29.73,41.40
2,test_2024,Same hour yesterday,8784,23.19,33.24
3,test_2024,Same hour last week,8784,28.23,40.96


## Select without touching the test result


In [16]:
validation_results = benchmark_results.query("split == 'validation_2023'").copy()
selected_benchmark = validation_results.sort_values("mae_eur_per_mwh").iloc[0]

final_test_result = benchmark_results.loc[
    (benchmark_results["split"] == "test_2024")
    & (benchmark_results["benchmark"] == selected_benchmark["benchmark"])
].iloc[0]

print(
    "Selected from validation only: "
    f"{selected_benchmark['benchmark']} "
    f"(MAE {selected_benchmark['mae_eur_per_mwh']:.2f} EUR/MWh)"
)
print(
    "Final 2024 test result: "
    f"MAE {final_test_result['mae_eur_per_mwh']:.2f} EUR/MWh, "
    f"RMSE {final_test_result['rmse_eur_per_mwh']:.2f} EUR/MWh"
)


Selected from validation only: Same hour yesterday (MAE 23.76 EUR/MWh)
Final 2024 test result: MAE 23.19 EUR/MWh, RMSE 33.24 EUR/MWh
